In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Silver Layer Script

### Data Access

In [0]:
spark.conf.set("fs.azure.account.auth.type.swap01storageaccount.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.swap01storageaccount.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.swap01storageaccount.dfs.core.windows.net", "***********************")
spark.conf.set("fs.azure.account.oauth2.client.secret.swap01storageaccount.dfs.core.windows.net", "88888888888888888888888888")
spark.conf.set("fs.azure.account.oauth2.client.endpoint.swap01storageaccount.dfs.core.windows.net", "https://login.microsoftonline.com/9999999999999999999999999999/oauth2/token")

### Data Loading

In [0]:
df_calendar = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Calendar")

In [0]:
df_customers = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Customers")

In [0]:
df_product_categories = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Product_Categories")

In [0]:
df_product_subcategories = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Product_Subcategories")


In [0]:
df_products = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Products")


In [0]:
df_returns = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Returns")


In [0]:
df_sales_2015 = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Sales_2015")


In [0]:
df_sales_2016 = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Sales_2016")


In [0]:
df_sales_2017 = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Sales_2017")


In [0]:
df_territories = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load("abfss://bronze@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Territories")

In [0]:
df_calendar.display()

df_customers.display()

df_product_categories.display()

df_product_subcategories.display()

df_products.display()

df_returns.display()

df_sales_2015.display()

df_sales_2016.display()

df_sales_2017.display()

df_territories.display()

### Transformations:

### Calender

In [0]:
df_calendar = df_calendar \
    .withColumn("Month", month(col("Date"))) \
    .withColumn("Year", year(col("Date"))) \
    .withColumn("Quarter", quarter(col("Date")))

df_calendar.display()

In [0]:
df_calendar.write.format('parquet')\
                 .mode('append')\
                 .option("path", "abfss://silver@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Calendar")\
                 .save()

### Customer

In [0]:
df_customers = df_customers.withColumn("FullName", concat_ws(' ', col("FirstName"), col("LastName")))

df_customers.display()

In [0]:
df_customers.write.format('parquet')\
                 .mode('append')\
                 .option("path", "abfss://silver@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Customers")\
                 .save()


### SUBCATEGORIES

In [0]:
df_product_subcategories.display()

In [0]:
df_product_subcategories.write.format('parquet')\
                 .mode('append')\
                 .option("path", "abfss://silver@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Product_Subcategories")\
                 .save()

### Product

In [0]:
df_products.display()

In [0]:
df_products = df_products.withColumn("ProductSKU", split(col("ProductSKU"),'-')[0])\
                        .withColumn("ModelName", split(col("ModelName"),'-')[0] )


df_products.display()

In [0]:
df_products.write.format('parquet')\
                 .mode('append')\
                 .option("path", "abfss://silver@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Products")\
                 .save()

### Product categories

In [0]:
df_product_categories.display()

In [0]:
df_product_categories.write.format('parquet')\
                 .mode('append')\
                 .option("path", "abfss://silver@swap01storageaccount.dfs.core.windows.net/AdventureWorks_product_categories")\
                 .save()

### Returns

In [0]:
df_returns.display()


In [0]:
df_returns.write.format('parquet')\
                 .mode('append')\
                 .option("path", "abfss://silver@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Returns")\
                 .save()

### Teroteries

In [0]:
df_territories.display()

In [0]:
df_territories.write.format('parquet')\
                 .mode('append')\
                 .option("path", "abfss://silver@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Territories")\
                 .save()

### Sales Transformations

In [0]:
df_sales = df_sales_2015 \
    .unionByName(df_sales_2016) \
    .unionByName(df_sales_2017)

df_sales.display()

In [0]:
df_sales = df_sales.withColumn("StockDate", to_timestamp('StockDate'))         

In [0]:
df_sales = df_sales.withColumn("OrderNumber", regexp_replace(col('OrderNumber'), 'SO', 'AWB'))

In [0]:
df_sales = df_sales.withColumn("Multiply", col('OrderLineItem')* col('OrderQuantity'))

In [0]:
df_sales.display()

## Sales Analysis

In [0]:
df_sales.groupBy('OrderDate').agg(count('OrderNumber').alias('Total Orders')).display()


Databricks visualization. Run in Databricks to view.

In [0]:
df_product_categories.display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_territories.display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_sales.write.format('parquet')\
                 .mode('append')\
                 .option("path", "abfss://silver@swap01storageaccount.dfs.core.windows.net/AdventureWorks_Sales")\
                 .save()